## Generating training dataset

In [ ]:
# --- STEP 1: Safe columns from Listings ---
# We ONLY take physical/static attributes. We avoid 'number_of_reviews' to prevent data leakage.
static_listing_cols = [
    'id', 'property_type', 'room_type', 'accommodates',
    'bathrooms_text', 'bedrooms', 'price', 'neighbourhood_cleansed'
]
safe_listings = listings_df[static_listing_cols].rename(columns={'id': 'listing_id'})

# Ensure listing_id types match for merging
safe_listings['listing_id'] = safe_listings['listing_id'].astype('Int64')
training_data['listing_id'] = training_data['listing_id'].astype('Int64')

In [ ]:
sessions_df_clean = sessions_df_clean[sessions_df_clean['action'].isin(['view_listing', 'book_listing'])]
sessions_sessions_df_clean = sessions_df_clean.drop_duplicates()   # drop duplicates to not destroy Y balance

# Seperate views from bookings
views_df = sessions_df_clean[sessions_df_clean["action"] == "view_listing"][['user_id', 'timestamp', 'listing_id']].copy()
bookings_df = sessions_df_clean[sessions_df_clean["action"] == "book_listing"][['user_id', 'listing_id', 'timestamp']].copy()
bookings_df = bookings_df.rename(columns={'timestamp': 'booking_timestamp'})

# Forward point-in-time join
views_df = views_df.sort_values('timestamp')
bookings_df = bookings_df.sort_values('booking_timestamp')
merged_df = pd.merge_asof(
    left=views_df,
    right=bookings_df,
    left_on='timestamp',
    right_on='booking_timestamp',
    by=['user_id', 'listing_id'],
    direction='forward',                # Events after T0
    tolerance=pd.Timedelta(hours=24)    # For consistency, session mustn't be longer than 24h
)

# If booking timestamp is not none, the offer was reserved
merged_df['Y'] = merged_df['booking_timestamp'].notna().astype(int)
training_data = merged_df.drop(columns=['booking_timestamp'])

In [ ]:
print(len(training_data[training_data["Y"] == 1]))
print(len(training_data[training_data["Y"] == 0]))

In [ ]:
training_data[training_data["user_id"] == 162656.0]

In [ ]:
multiple_y1 = training_data[training_data['Y'] == 1].groupby(['user_id', 'listing_id']).size()
print(f"Reservations with viewing multiple times: {(multiple_y1 > 1).sum()}")
multiple_y1[multiple_y1 > 1].head()

In [ ]:
def calculate_historical_features(row):
    t0 = row['timestamp']
    l_id = row['listing_id']

    mask_listing = (reviews_df_clean['listing_id'] == l_id)
    mask_past = (reviews_df_clean['date'] < t0)
    mask_recent = (reviews_df_clean['date'] >= (t0 - pd.Timedelta(days=7))) # Last 7 days

    recent_reviews = reviews_df_clean[mask_listing & mask_past & mask_recent]

    # Calculated metrics
    reviews_count_7d = len(recent_reviews)
    days_from_last_review = ...

    # Generated metrics
    prompt = """Generate response in format {...} for this comment: {...}
"""
    score_metrics_response = {}
    # recent_sentiment_avg = recent_reviews['nlp_sentiment_score'].mean() if not recent_reviews.empty else 0.0

    # Zwracamy wyliczone wartości jako nowe kolumny
    return pd.Series({
        'reviews_in_last_7_days': reviews_count_7d
        'days_from_last_review': days_from_last_review,
        **response
    })

# Uruchamiamy obliczenia dla każdego wiersza w naszym zbiorze treningowym
# Uwaga: metoda .apply() wykonuje się wiersz po wierszu.
features_df = training_data.apply(calculate_historical_features, axis=1)

# Łączymy wygenerowane cechy z naszą główną tabelą
final_training_dataset = pd.concat([training_data, features_df], axis=1)

# Wynik gotowy do uczenia modelu
print(final_training_dataset.head())

## Class distribution

## Generating inference sample